In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

# Load dataset
df = pd.read_csv("../data/synthetic/orders_data.csv")

df.head()

,order_id,city,category,customer_type,order_time,distance_km,order_value,delivery_fee,rider_cost,dark_store_cost,discount,delivery_time,gross_margin,contribution_margin,peak_hour
0,1,Chennai,Groceries,Discount-Seeker,2025-03-23 04:18:00,0.78,556.59,33.44,36.64,38.53,61.42,13.06,93.744181,-9.41,0
1,2,Pune,Groceries,Premium,2025-06-05 06:02:00,5.08,589.88,21.22,63.54,23.04,56.31,27.63,125.453680,3.78,0
2,3,Mumbai,Groceries,Premium,2025-02-26 02:07:00,4.55,574.73,21.45,63.81,48.23,0.64,19.84,90.009690,-1.22,0
3,4,Hyderabad,Groceries,Regular,2025-01-23 18:38:00,2.26,419.96,37.76,52.14,40.46,3.17,15.97,67.792234,9.78,0
4,5,Chennai,Beauty,Discount-Seeker,2025-05-06 16:17:00,5.84,324.27,40.76,71.86,45.38,51.73,24.87,61.538436,-66.67,0


In [2]:
# Generate synthetic customer IDs

num_customers = 4000

df["customer_id"] = np.random.randint(
    1,
    num_customers + 1,
    size=len(df)
)

df.head()

,order_id,city,category,customer_type,order_time,distance_km,order_value,delivery_fee,rider_cost,dark_store_cost,discount,delivery_time,gross_margin,contribution_margin,peak_hour,customer_id
0,1,Chennai,Groceries,Discount-Seeker,2025-03-23 04:18:00,0.78,556.59,33.44,36.64,38.53,61.42,13.06,93.744181,-9.41,0,2774
1,2,Pune,Groceries,Premium,2025-06-05 06:02:00,5.08,589.88,21.22,63.54,23.04,56.31,27.63,125.453680,3.78,0,3291
2,3,Mumbai,Groceries,Premium,2025-02-26 02:07:00,4.55,574.73,21.45,63.81,48.23,0.64,19.84,90.009690,-1.22,0,2625
3,4,Hyderabad,Groceries,Regular,2025-01-23 18:38:00,2.26,419.96,37.76,52.14,40.46,3.17,15.97,67.792234,9.78,0,3005
4,5,Chennai,Beauty,Discount-Seeker,2025-05-06 16:17:00,5.84,324.27,40.76,71.86,45.38,51.73,24.87,61.538436,-66.67,0,1354


In [3]:
customer_metrics = df.groupby("customer_id").agg({

    "order_id": "count",

    "order_value": "mean",

    "contribution_margin": "sum",

    "delivery_time": "mean"

}).reset_index()

customer_metrics.columns = [

    "customer_id",
    "total_orders",
    "avg_order_value",
    "total_profit",
    "avg_delivery_time"
]

customer_metrics.head()

,customer_id,total_orders,avg_order_value,total_profit,avg_delivery_time
0,1,3,424.836667,-131.38,18.360000
1,2,7,436.302857,88.13,17.570000
2,3,5,501.090000,-53.83,22.326000
3,4,1,367.320000,28.75,19.740000
4,5,11,441.531818,-227.50,20.051818


In [4]:
# Segment customers

conditions = [

    (customer_metrics["total_orders"] >= 10)
    &
    (customer_metrics["avg_order_value"] >= 500),

    (customer_metrics["total_orders"] >= 5),

    (customer_metrics["total_orders"] < 5)

]

segments = [
    "Premium Loyalists",
    "Regular Users",
    "Low Engagement"
]

customer_metrics["segment"] = np.select(
    conditions,
    segments,
    default="Others"
)

customer_metrics.head()

,customer_id,total_orders,avg_order_value,total_profit,avg_delivery_time,segment
0,1,3,424.836667,-131.38,18.360000,Low Engagement
1,2,7,436.302857,88.13,17.570000,Regular Users
2,3,5,501.090000,-53.83,22.326000,Regular Users
3,4,1,367.320000,28.75,19.740000,Low Engagement
4,5,11,441.531818,-227.50,20.051818,Regular Users


In [5]:
segment_analysis = customer_metrics.groupby("segment").agg({

    "customer_id": "count",

    "total_orders": "mean",

    "avg_order_value": "mean",

    "total_profit": "mean"

}).reset_index()

segment_analysis

,segment,customer_id,total_orders,avg_order_value,total_profit
0,Low Engagement,1721,3.018594,450.521431,-35.766008
1,Premium Loyalists,8,11.125000,513.480811,-13.635000
2,Regular Users,2249,6.543353,449.174467,-79.252259


In [6]:
fig = px.bar(

    segment_analysis,

    x="segment",

    y="total_profit",

    color="segment",

    title="Customer Segment Profitability"

)

fig.show()

In [9]:
# Create absolute profit column for bubble sizing

customer_metrics["profit_size"] = (
    customer_metrics["total_profit"].abs()
)

fig = px.scatter(

    customer_metrics,

    x="total_orders",

    y="avg_order_value",

    color="segment",

    size="profit_size",

    opacity=0.7,

    title="Customer Order Frequency vs Basket Value"

)

fig.show()